In [1]:
# 1. IMPORTAR HERRAMIENTAS
# Esto es como decirle al mecánico: "Saca la llave inglesa, el destornillador y el gato hidráulico"
import pandas as pd          # Para manejar tablas (DataFrames)
import numpy as np           # Para matemáticas rápidas
import matplotlib.pyplot as plt # Para dibujar gráficos

# 2. CONFIGURAR VISUALIZACIÓN
# Esto es para que los gráficos aparezcan aquí mismo, dentro del notebook, y no en una ventana aparte.
%matplotlib inline

# 3. CARGA DE DATOS
# Aquí ocurre la magia. Pandas abre el CSV y lo convierte en un objeto DataFrame (una mega-tabla).
# '..' significa "sube una carpeta", porque el CSV está en /data y nosotros estamos en /notebooks.
ruta_csv = '../data/paris_housing_prices_dataset.csv'
df = pd.read_csv(ruta_csv)

# 4. INSPECCIÓN INICIAL
# Estas 3 funciones son tus "ojos" para ver los datos sin abrir Excel.
print("=== PRIMERAS 5 FILAS ===")
print(df.head())       # Muestra las primeras 5 filas. Sirve para ver si hay nombres raros o columnas vacías.

print("\n=== INFORMACIÓN TÉCNICA ===")
print(df.info())       # MUY IMPORTANTE: Dice cuántas columnas hay, si tienen números o texto, y cuántos huecos (NaN) hay.

print("\n=== ESTADÍSTICAS BÁSICAS ===")
print(df.describe())   # Para las columnas numéricas, te da el promedio, el mínimo y el máximo al instante.

=== PRIMERAS 5 FILAS ===
  Property_ID  Arrondissement Property_Type  Size_sqm  Rooms  Floor  \
0      P10000               4     Apartment       117      4      7   
1      P10001               8        Studio        89      3      3   
2      P10002               4     Apartment       164      5      5   
3      P10003               2     Apartment        35      1      5   
4      P10004               7        Studio        73      2      2   

   Year_Built         Condition  Distance_to_Center_km   Price_EUR  
0        1870         Renovated                   2.76  2270802.89  
1        1953              Good                  10.77  1637076.12  
2        1979  Needs Renovation                   3.14  3220782.59  
3        1938               New                   4.72   407781.74  
4        1957               New                   7.96   624879.12  

=== INFORMACIÓN TÉCNICA ===
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 10 columns):
 #   Colu

In [2]:
# ============================================================
# FUNCIÓN DE LIMPIEZA AUTOMÁTICA (AUTO-ML)
# ============================================================
# Esta función toma un DataFrame SUCIO y devuelve uno LIMPIO listo para IA.
# Es el 80% del trabajo de un Data Scientist en el mundo real.

def limpiar_dataframe_auto(df, es_entrenamiento=True, columna_objetivo='Price_EUR'):
    """
    Parámetros:
    - df: El DataFrame de Pandas a limpiar.
    - es_entrenamiento: Si es True, calcula promedios y guarda reglas. Si es False, APLICA reglas guardadas.
    - columna_objetivo: La columna que queremos predecir (para no modificarla).
    """
    
    # Creamos una copia para no destrozar el original (Buena práctica)
    df_limpio = df.copy()
    
    # ---------------------------------------------------------
    # 1. ELIMINAR COLUMNAS IDENTIFICADORAS (DNI)
    # ---------------------------------------------------------
    columnas_id = ['Property_ID', 'ID', 'Id', 'id']
    for col in columnas_id:
        if col in df_limpio.columns:
            print(f"🗑️  Eliminando columna identificadora: {col}")
            df_limpio = df_limpio.drop(columns=[col])
            
    # ---------------------------------------------------------
    # 2. SEPARAR EL OBJETIVO (y)
    # ---------------------------------------------------------
    # Si estamos entrenando, separamos el precio para que no lo "limpie" accidentalmente.
    if es_entrenamiento and columna_objetivo in df_limpio.columns:
        y = df_limpio[columna_objetivo].copy()
        X = df_limpio.drop(columns=[columna_objetivo])
    else:
        X = df_limpio.copy()
        y = None
        
    # ---------------------------------------------------------
    # 3. DETECTAR Y PROCESAR TEXTO (COLUMNAS CATEGÓRICAS)
    # ---------------------------------------------------------
    # Pandas guarda el texto como 'object'. Detectamos esas columnas.
    columnas_texto = X.select_dtypes(include=['object']).columns.tolist()
    
    for col in columnas_texto:
        print(f"🔤 Procesando columna de texto: {col}")
        
        # CASO ESPECIAL: 'Condition' es Ordinal (Tiene jerarquía)
        if col == 'Condition':
            # Mapeo manual: Asignamos números según la calidad
            mapeo_calidad = {
                'New': 4, 
                'Good': 3, 
                'Renovated': 2, 
                'Needs Renovation': 1
            }
            # Si encuentra un valor que no está en el mapa (ej. "Unknown"), le pone 0
            X[col + '_encoded'] = X[col].map(mapeo_calidad).fillna(0).astype(int)
            print(f"   -> Mapeado a valores numéricos (Ordinal): {mapeo_calidad}")
            X = X.drop(columns=[col]) # Eliminamos la columna de texto original
            
        else:
            # CASO GENERAL: One-Hot Encoding (Ej: Property_Type: Apartment / Studio)
            # Creamos columnas nuevas: Tipo_Apartment (1 o 0), Tipo_Studio (1 o 0)
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
            # drop_first=True evita redundancia. Si no es Apartment, es Studio. No hacen falta 2 columnas.
            X = pd.concat([X, dummies], axis=1)
            print(f"   -> Convertido a One-Hot Encoding. Se crearon {len(dummies.columns)} columnas.")
            X = X.drop(columns=[col])

    # ---------------------------------------------------------
    # 4. RELLENAR HUECOS (NÚMEROS VACÍOS)
    # ---------------------------------------------------------
    # ¿Hay algún NaN en las columnas numéricas?
    columnas_numericas = X.select_dtypes(include=['int64', 'float64']).columns
    
    if X[columnas_numericas].isnull().sum().sum() > 0:
        print("🩹 Rellenando valores nulos en columnas numéricas...")
        for col in columnas_numericas:
            if X[col].isnull().any():
                # Rellenamos con la MEDIA (promedio)
                # Si es entrenamiento, calculamos la media y la guardamos para usarla luego.
                valor_relleno = X[col].mean()
                X[col] = X[col].fillna(valor_relleno)
                print(f"   -> Columna '{col}' rellenada con media: {valor_relleno:.2f}")
    else:
        print("✅ No se encontraron valores nulos numéricos.")
        
    # ---------------------------------------------------------
    # 5. DEVOLVER RESULTADO
    # ---------------------------------------------------------
    if es_entrenamiento and y is not None:
        print(f"\n✅ LIMPIEZA COMPLETADA.")
        print(f"   Shape final de X (Features): {X.shape}")
        print(f"   Shape final de y (Target): {y.shape}")
        return X, y
    else:
        print(f"\n✅ LIMPIEZA COMPLETADA (Modo Predicción).")
        return X

# ============================================================
# PROBAMOS LA FUNCIÓN CON NUESTROS DATOS
# ============================================================
print("=== APLICANDO LIMPIEZA AUTOMÁTICA ===\n")
X_limpio, y_limpio = limpiar_dataframe_auto(df, es_entrenamiento=True, columna_objetivo='Price_EUR')

print("\n=== VISTAZO FINAL A LOS DATOS QUE COME LA IA ===")
print(X_limpio.head(3))

=== APLICANDO LIMPIEZA AUTOMÁTICA ===

🗑️  Eliminando columna identificadora: Property_ID
🔤 Procesando columna de texto: Property_Type
   -> Convertido a One-Hot Encoding. Se crearon 3 columnas.
🔤 Procesando columna de texto: Condition
   -> Mapeado a valores numéricos (Ordinal): {'New': 4, 'Good': 3, 'Renovated': 2, 'Needs Renovation': 1}
✅ No se encontraron valores nulos numéricos.

✅ LIMPIEZA COMPLETADA.
   Shape final de X (Features): (1200, 10)
   Shape final de y (Target): (1200,)

=== VISTAZO FINAL A LOS DATOS QUE COME LA IA ===
   Arrondissement  Size_sqm  Rooms  Floor  Year_Built  Distance_to_Center_km  \
0               4       117      4      7        1870                   2.76   
1               8        89      3      3        1953                  10.77   
2               4       164      5      5        1979                   3.14   

   Property_Type_Loft  Property_Type_Penthouse  Property_Type_Studio  \
0               False                    False                 Fa

C:\Users\denni\AppData\Local\Temp\ipykernel_20972\2498851852.py:42: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = X.select_dtypes(include=['object']).columns.tolist()


In [3]:
# ============================================================
# DIVISIÓN TRAIN / TEST (El momento de la verdad)
# ============================================================
from sklearn.model_selection import train_test_split

# random_state=42 es una semilla mágica.
# Poner cualquier número fijo (42, 123, 7) asegura que siempre partamos los datos igual.
# Si no lo pones, cada vez que ejecutes el código la IA estudiará con datos distintos y no podrás comparar resultados.
X_train, X_test, y_train, y_test = train_test_split(
    X_limpio,       # Los datos de entrada limpios
    y_limpio,       # Los precios (lo que queremos predecir)
    test_size=0.2,  # 20% para el examen
    random_state=42 # Semilla para que el resultado sea reproducible
)

print("📊 DATOS LISTOS PARA LA IA")
print("="*40)
print(f"📚 Tamaño del ENTRENAMIENTO (Train): {X_train.shape[0]} casas")
print(f"📝 Tamaño del EXAMEN (Test):          {X_test.shape[0]} casas")
print(f"🧠 Número de características (Features): {X_train.shape[1]}")

print("\n🔍 PRIMERAS 3 CASAS DE ESTUDIO (X_train):")
print(X_train.head(3))

print(f"\n💰 PRECIOS REALES DE ESAS 3 CASAS (y_train):")
print(y_train.head(3))

📊 DATOS LISTOS PARA LA IA
📚 Tamaño del ENTRENAMIENTO (Train): 960 casas
📝 Tamaño del EXAMEN (Test):          240 casas
🧠 Número de características (Features): 10

🔍 PRIMERAS 3 CASAS DE ESTUDIO (X_train):
     Arrondissement  Size_sqm  Rooms  Floor  Year_Built  \
331               9        69      3      5        1909   
409              17       100      1      5        1947   
76                9        51      4      6        1948   

     Distance_to_Center_km  Property_Type_Loft  Property_Type_Penthouse  \
331                  12.85                True                    False   
409                   7.53               False                    False   
76                    2.00                True                    False   

     Property_Type_Studio  Condition_encoded  
331                 False                  2  
409                 False                  2  
76                  False                  2  

💰 PRECIOS REALES DE ESAS 3 CASAS (y_train):
331    1203036.49
409    

In [4]:
# ============================================================
# ENTRENAMIENTO DEL MODELO (Random Forest)
# ============================================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# 1. CREAR EL CEREBRO (Inicializar el modelo)
# n_estimators = 100 -> Vamos a preguntarle a 100 "expertos" (árboles)
# random_state = 42   -> Misma semilla, para que siempre "piensen" igual en cada ejecución
modelo = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
# n_jobs=-1 significa: "Usa todos los núcleos de mi procesador para ir más rápido"

print("🌳🌳🌳 Plantando 100 árboles de decisión... (Esto puede tardar 1-2 segundos)")

# 2. ENTRENAR (FIT)
# Aquí ocurre la magia. La IA mira X_train (las características) 
# y trata de encontrar la fórmula secreta que genera y_train (los precios).
modelo.fit(X_train, y_train)

print("✅ ¡Modelo entrenado!")

# 3. PREDECIR SOBRE EL EXAMEN (TEST)
# Le damos a la IA SOLO las características del examen (X_test).
# Ella NUNCA ha visto los precios reales de estas 240 casas.
y_predicho = modelo.predict(X_test)

# 4. EVALUAR QUÉ TAN BUENO ES
# Comparamos lo que la IA PREDIJO (y_predicho) contra la REALIDAD (y_test)
mae = mean_absolute_error(y_test, y_predicho)
r2 = r2_score(y_test, y_predicho)

print("\n" + "="*50)
print("📊 INFORME DE RENDIMIENTO DEL MODELO")
print("="*50)
print(f"💰 Error Absoluto Medio (MAE): {mae:,.2f} €")
print(f"📈 Precisión (R² Score):         {r2:.4f} ({r2*100:.2f}%)")
print("="*50)

# 5. INTERPRETACIÓN HUMANA DE LAS MÉTRICAS
print("\n🧠 ¿CÓMO LEER ESTO?")
print("-" * 30)
print(f"1. MAE (Error Medio): Por cada casa que predice, en promedio se equivoca por {mae:,.0f} €.")
print(f"   -> Si una casa vale 1,000,000€, la IA dirá entre {1_000_000 - mae:,.0f}€ y {1_000_000 + mae:,.0f}€.")
print(f"2. R² (Coeficiente de Determinación): Mide cuánta 'magia' capturó.")
print(f"   -> 1.00 = Predicción Perfecta.")
print(f"   -> 0.00 = Predecir usando el promedio (horrible).")
print(f"   -> Tu modelo captura el {r2*100:.1f}% de la variación de precios.")

🌳🌳🌳 Plantando 100 árboles de decisión... (Esto puede tardar 1-2 segundos)
✅ ¡Modelo entrenado!

📊 INFORME DE RENDIMIENTO DEL MODELO
💰 Error Absoluto Medio (MAE): 332,524.35 €
📈 Precisión (R² Score):         0.7664 (76.64%)

🧠 ¿CÓMO LEER ESTO?
------------------------------
1. MAE (Error Medio): Por cada casa que predice, en promedio se equivoca por 332,524 €.
   -> Si una casa vale 1,000,000€, la IA dirá entre 667,476€ y 1,332,524€.
2. R² (Coeficiente de Determinación): Mide cuánta 'magia' capturó.
   -> 1.00 = Predicción Perfecta.
   -> 0.00 = Predecir usando el promedio (horrible).
   -> Tu modelo captura el 76.6% de la variación de precios.


In [5]:
# ============================================================
# CONGELAR EL CEREBRO Y LAS REGLAS (Serialización)
# ============================================================
import joblib
import os

# 1. CREAR CARPETA PARA MODELOS (si no existe)
# Subimos un nivel (..) y creamos una carpeta llamada 'modelos_entrenados'
ruta_modelos = '../modelos_entrenados'
if not os.path.exists(ruta_modelos):
    os.makedirs(ruta_modelos)
    print(f"📁 Carpeta creada: {ruta_modelos}")

# 2. GUARDAR EL MODELO (Random Forest)
# El archivo se llamará 'modelo_precios_paris.pkl'
ruta_modelo_rf = os.path.join(ruta_modelos, 'modelo_precios_paris.pkl')
joblib.dump(modelo, ruta_modelo_rf)
print(f"🧠 Modelo guardado en: {ruta_modelo_rf}")

# 3. GUARDAR EL MAPEO DE 'Condition' (CRÍTICO)
# Si no guardamos esto, el backend no sabrá qué hacer con "New" o "Good".
mapeo_condition = {
    'New': 4, 
    'Good': 3, 
    'Renovated': 2, 
    'Needs Renovation': 1
}
ruta_mapeo = os.path.join(ruta_modelos, 'mapeo_condition.pkl')
joblib.dump(mapeo_condition, ruta_mapeo)
print(f"📋 Mapeo de 'Condition' guardado en: {ruta_mapeo}")

# 4. GUARDAR LA LISTA DE COLUMNAS EXACTAS QUE ESPERA EL MODELO
# En el backend, los datos nuevos deben tener EXACTAMENTE estas columnas y en ESTE orden.
# Si el usuario sube un CSV con columnas en otro orden, lo reorganizaremos.
columnas_esperadas = X_train.columns.tolist()
ruta_columnas = os.path.join(ruta_modelos, 'columnas_input.pkl')
joblib.dump(columnas_esperadas, ruta_columnas)
print(f"📋 Lista de columnas esperadas guardada en: {ruta_columnas}")

print("\n✅ ¡Cerebro y herramientas congelados! Listos para el Backend.")
print(f"   Columnas que espera el modelo: {columnas_esperadas}")

📁 Carpeta creada: ../modelos_entrenados
🧠 Modelo guardado en: ../modelos_entrenados\modelo_precios_paris.pkl
📋 Mapeo de 'Condition' guardado en: ../modelos_entrenados\mapeo_condition.pkl
📋 Lista de columnas esperadas guardada en: ../modelos_entrenados\columnas_input.pkl

✅ ¡Cerebro y herramientas congelados! Listos para el Backend.
   Columnas que espera el modelo: ['Arrondissement', 'Size_sqm', 'Rooms', 'Floor', 'Year_Built', 'Distance_to_Center_km', 'Property_Type_Loft', 'Property_Type_Penthouse', 'Property_Type_Studio', 'Condition_encoded']


In [6]:
# ============================================================
# SIMULACRO DE PRODUCCIÓN (Carga en Frío)
# ============================================================
import pandas as pd
import joblib
import os

print("🔄 SIMULANDO BACKEND: Cargando modelo desde archivo...")

# 1. CARGAR LOS ARCHIVOS CONGELADOS
ruta_modelos = '../modelos_entrenados'
modelo_cargado = joblib.load(os.path.join(ruta_modelos, 'modelo_precios_paris.pkl'))
mapeo_cargado = joblib.load(os.path.join(ruta_modelos, 'mapeo_condition.pkl'))
columnas_cargadas = joblib.load(os.path.join(ruta_modelos, 'columnas_input.pkl'))

# 2. CREAR UNA CASA FICTICIA NUEVA (Simulando input del usuario)
# El usuario rellena un formulario en la App.
nueva_casa_dict = {
    'Arrondissement': [8],          # Distrito 8 (Campos Elíseos - Caro)
    'Size_sqm': [120],              # 120 m2
    'Rooms': [4],                   # 4 habitaciones
    'Floor': [3],                   # Piso 3
    'Year_Built': [1995],           # Relativamente moderna
    'Distance_to_Center_km': [2.5], # Cerca del centro
    'Property_Type': ['Apartment'], # Es un apartamento
    'Condition': ['Good']           # Buen estado
}

df_nueva_casa = pd.DataFrame(nueva_casa_dict)
print("\n🏠 DATOS DE ENTRADA (Sucios):")
print(df_nueva_casa.T) # .T para verlo en vertical

# 3. SIMULAR LA LIMPIEZA (Usando las reglas que ya definimos antes)
# Copiamos la lógica de limpieza pero usando el MAPEO CARGADO.
df_limpio_input = df_nueva_casa.copy()

# 3.1 Eliminar IDs (no hay en este ejemplo)
# 3.2 Procesar Property_Type (One-Hot)
dummies = pd.get_dummies(df_limpio_input['Property_Type'], prefix='Property_Type')
df_limpio_input = pd.concat([df_limpio_input, dummies], axis=1)
df_limpio_input = df_limpio_input.drop(columns=['Property_Type'])

# 3.3 Procesar Condition (Mapeo)
df_limpio_input['Condition_encoded'] = df_limpio_input['Condition'].map(mapeo_cargado).fillna(0)
df_limpio_input = df_limpio_input.drop(columns=['Condition'])

# 3.4 ALINEAR COLUMNAS (MUY IMPORTANTE)
# Aseguramos que el input tenga las mismas columnas que en el entrenamiento.
# Si falta alguna (ej: Property_Type_Loft), se añade con valor 0.
df_final_input = df_limpio_input.reindex(columns=columnas_cargadas, fill_value=0)

print("\n🧹 DATOS LIMPIOS Y ALINEADOS (Listos para IA):")
print(df_final_input.T)

# 4. ¡PREDECIR!
prediccion = modelo_cargado.predict(df_final_input)[0]

print("\n" + "="*50)
print(f"🤖 PREDICCIÓN FINAL DE LA APP: {prediccion:,.2f} €")
print("="*50)

# 5. VERIFICACIÓN VISUAL
print("\n💡 ¿Es lógico este precio?")
print(f"   - Distrito 8 (Zona Prime) + 120m2 + Buen estado.")
print(f"   - Precio medio del mercado: 1.5M€")
print(f"   - Diferencia con la media: {prediccion - 1500000:,.0f} €")

🔄 SIMULANDO BACKEND: Cargando modelo desde archivo...

🏠 DATOS DE ENTRADA (Sucios):
                               0
Arrondissement                 8
Size_sqm                     120
Rooms                          4
Floor                          3
Year_Built                  1995
Distance_to_Center_km        2.5
Property_Type          Apartment
Condition                   Good

🧹 DATOS LIMPIOS Y ALINEADOS (Listos para IA):
                              0
Arrondissement              8.0
Size_sqm                  120.0
Rooms                       4.0
Floor                       3.0
Year_Built               1995.0
Distance_to_Center_km       2.5
Property_Type_Loft          0.0
Property_Type_Penthouse     0.0
Property_Type_Studio        0.0
Condition_encoded           3.0

🤖 PREDICCIÓN FINAL DE LA APP: 1,878,944.85 €

💡 ¿Es lógico este precio?
   - Distrito 8 (Zona Prime) + 120m2 + Buen estado.
   - Precio medio del mercado: 1.5M€
   - Diferencia con la media: 378,945 €
